---
# 1. Environment Setup and Google Drive Mount

In this section, we:

- Import all required libraries for:
  - Numerical computing (`numpy`, `pandas`)
  - Deep learning (`tensorflow.keras`)
  - Hyperparameter tuning (`keras_tuner`)
  - Visualisation (`matplotlib`)
- Mount Google Drive to access the processed CV dataset and to save
  all trained models, histories, and plots persistently.
- Define base directories for:
  - Processed data (`cv_data_processed`)
  - Model artefacts (`cv_models`)
  - Training curves and logs.

In [ ]:
!pip install -q keras-tuner

"""Environment setup and global configuration for CV model training.

This module:
    • Mounts Google Drive in the Colab environment.
    • Configures global directories for processed data and model artefacts.
    • Logs basic environment information (e.g., TensorFlow version).
"""

from __future__ import annotations

import json
import os
from typing import Dict, List, Sequence, Tuple

import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive  # type: ignore


def mount_google_drive() -> None:
    """Mount Google Drive in the Colab environment.

    This function mounts the user's Google Drive at ``/content/drive`` so
    that datasets, models, and artefacts can be read from and written to
    persistent storage.

    Returns:
        None. The function performs the side effect of mounting Drive and
        prints a confirmation message.
    """
    drive.mount("/content/drive")
    print("[INFO] Google Drive mounted at /content/drive.")


# ---------------------------------------------------------------------------
# Initialisation
# ---------------------------------------------------------------------------
mount_google_drive()

# ---------------------------------------------------------------------------
# Global configuration
# ---------------------------------------------------------------------------
RANDOM_STATE: int = 42

# NOTE: Adjust this if your processed data is in a different location.
PROCESSED_DATA_DIR: str = (
    "/content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed"
)

# Directory to store models, histories, and plots.
MODEL_ROOT_DIR: str = (
    "/content/drive/MyDrive/Colab Notebooks/DOAA/cv_models"
)
os.makedirs(MODEL_ROOT_DIR, exist_ok=True)

print(f"[INFO] Processed data directory: {PROCESSED_DATA_DIR}")
print(f"[INFO] Model artefacts directory: {MODEL_ROOT_DIR}")

# Basic TensorFlow info
print("[INFO] TensorFlow version:", tf.__version__)

Mounted at /content/drive
[INFO] Google Drive mounted at /content/drive.
[INFO] Processed data directory: /content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed
[INFO] Model artefacts directory: /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models
[INFO] TensorFlow version: 2.19.0


- Google Drive is successfully mounted.
- The notebook now knows where to find:
  - The processed CV data: `cv_data_processed`
  - The model artefacts: `cv_models`
- TensorFlow and supporting libraries are imported and ready.

Next, we will load the **train/validation/test** splits from the
processed dataset in Google Drive.

---
# 2. Loading Processed Train/Validation/Test Splits

In this section, we:

- Define helper functions to load the `labels.csv` files for each split:
  - `train/labels.csv`
  - `val/labels.csv`
  - `test/labels.csv`
- Attach the full `image_path` for each filename in the labels.
- Inspect the first few rows to confirm that:
  - Filenames are correctly mapped to their `images/` directory.
  - Target values (e.g. `price_usd`) are loaded properly.

In [ ]:
"""Utilities for loading image paths and labels for each CV split."""


def load_split_labels(
    base_dir: str,
    split_name: Literal["train", "val", "test"],
    target_col: str = "price_usd",
) -> pd.DataFrame:
    """Load labels for a given split and attach full image paths.

    Args:
        base_dir: Root directory of the processed dataset containing split
            subdirectories (for example, ``train/``, ``val/``, ``test/``).
        split_name: Name of the split to load. Must be one of
            ``'train'``, ``'val'``, or ``'test'``.
        target_col: Name of the target column in ``labels.csv``.

    Returns:
        A DataFrame with the following columns:
            - ``filename``: Image filename.
            - ``target_col``: Target variable (e.g., price in USD).
            - ``image_path``: Absolute path to the image file.

    Raises:
        FileNotFoundError: If ``labels.csv`` is missing for the split.
        NotADirectoryError: If the expected ``images/`` directory is missing.
        KeyError: If required columns are missing from ``labels.csv``.
    """
    split_dir = os.path.join(base_dir, split_name)
    images_dir = os.path.join(split_dir, "images")
    labels_path = os.path.join(split_dir, "labels.csv")

    if not os.path.isfile(labels_path):
        raise FileNotFoundError(f"labels.csv not found at: {labels_path}")

    if not os.path.isdir(images_dir):
        raise NotADirectoryError(
            f"Images directory not found at: {images_dir}"
        )

    df_labels = pd.read_csv(labels_path)

    required_cols = {"filename", target_col}
    missing_cols = required_cols.difference(df_labels.columns)
    if missing_cols:
        raise KeyError(
            f"Missing expected columns in labels.csv: {sorted(missing_cols)}"
        )

    df_labels["image_path"] = df_labels["filename"].apply(
        lambda name: os.path.join(images_dir, str(name))
    )

    # Keep only rows where the corresponding image file actually exists.
    df_labels = df_labels[df_labels["image_path"].apply(os.path.isfile)]

    print(
        f"[INFO] Loaded {len(df_labels)} samples for split '{split_name}' "
        f"from {labels_path}"
    )

    return df_labels.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Load all splits
# ---------------------------------------------------------------------------
df_train = load_split_labels(PROCESSED_DATA_DIR, "train")
df_val = load_split_labels(PROCESSED_DATA_DIR, "val")
df_test = load_split_labels(PROCESSED_DATA_DIR, "test")

print("\n[INFO] Sample of training labels:")
df_train.head()

[INFO] Loaded 10619 samples for split 'train' from /content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed/train/labels.csv
[INFO] Loaded 2276 samples for split 'val' from /content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed/val/labels.csv
[INFO] Loaded 2276 samples for split 'test' from /content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed/test/labels.csv

[INFO] Sample of training labels:


,filename,price_usd,image_path
0,29413668_f028ca049dc2194deb3febc33fa9c415-p_f.jpg,132500.0,/content/drive/MyDrive/Colab Notebooks/DOAA/cv...
1,83828234_485d8e0bcdbbe2f2e8fa5595a8d90e24-p_f.jpg,165000.0,/content/drive/MyDrive/Colab Notebooks/DOAA/cv...
2,29421441_da0cf48d2fc106913a9801e60bc7add7-p_f.jpg,250000.0,/content/drive/MyDrive/Colab Notebooks/DOAA/cv...
3,29436799_359ee1450f337abe63f59584d9acdea4-p_f.jpg,174900.0,/content/drive/MyDrive/Colab Notebooks/DOAA/cv...
4,29488470_2cb034e9cced90cd04b0365ca1aba564-p_f.jpg,349997.0,/content/drive/MyDrive/Colab Notebooks/DOAA/cv...


- `df_train`, `df_val`, and `df_test` now contain:
  - `filename`
  - `price_usd` (or the preprocessed target used in data prep)
  - `image_path` pointing to the actual PNG files.
- Non-existent image paths (if any) have been filtered out.

Next, we will transform these dataframes into TensorFlow `tf.data`
pipelines to drive the CNN training.


---
# 3. TensorFlow Data Pipelines for Image Regression

To efficiently train CNN models, we now:

- Define hyperparameters for:
  - Image height and width
  - Batch size
- Implement:
  - `load_and_preprocess_image()`:
    - Reads the image from disk
    - Decodes and resizes it
    - Scales pixel values to `[0, 1]`
  - `make_dataset_from_df()`:
    - Converts a labels DataFrame into a `tf.data.Dataset`
    - Applies batching, shuffling (for training), and prefetching

The resulting datasets:
- `train_ds`
- `val_ds`
- `test_ds`

will be used for baseline training and Keras Tuner.

In [ ]:
"""Image loading and tf.data pipeline utilities for CV house price regression.

This module:
    • Configures image shape and batching parameters.
    • Provides a robust image loader that handles corrupted files safely.
    • Builds tf.data.Dataset objects from label DataFrames.
"""


# ---------------------------------------------------------------------------
# Image and batching configuration
# ---------------------------------------------------------------------------
IMG_HEIGHT: int = 224
IMG_WIDTH: int = 224
BATCH_SIZE: int = 32


def load_and_preprocess_image(
    image_path: tf.Tensor,
    img_height: int,
    img_width: int,
) -> tf.Tensor:
    """Load an image from a file path and apply robust preprocessing.

    This function uses Pillow via a ``tf.numpy_function`` wrapper to safely
    handle corrupted or truncated image files.

    Behaviour:
        * On success:
            - Loads the image as RGB.
            - Resizes to (img_height, img_width).
            - Scales pixel values to [0.0, 1.0].
        * On failure (for example, invalid/corrupted image):
            - Logs a warning.
            - Returns a black image of the target size.

    Args:
        image_path: Tensor containing the filesystem path to an image.
        img_height: Target height for resizing.
        img_width: Target width for resizing.

    Returns:
        Preprocessed image tensor of shape
        ``(img_height, img_width, 3)`` with float32 values in [0, 1].
    """

    def _py_load_image(path_bytes: bytes) -> np.ndarray:
        """Python-side image loader using Pillow with error handling.

        Args:
            path_bytes: Byte string representing the image file path.

        Returns:
            A NumPy array of shape (img_height, img_width, 3) with float32
            values in [0, 1]. If loading fails, returns a black image.
        """
        # Imported here to avoid overhead at graph construction time.
        from PIL import Image  # type: ignore

        path_str = path_bytes.decode("utf-8")

        try:
            with Image.open(path_str) as img:
                img = img.convert("RGB")
                img = img.resize((img_width, img_height))
                arr = np.asarray(img, dtype="float32") / 255.0
        except Exception as exc:  # noqa: BLE001
            # Fallback: return a black image instead of crashing the pipeline.
            print(f"[WARN] Failed to load image '{path_str}': {exc}")
            arr = np.zeros((img_height, img_width, 3), dtype="float32")

        return arr

    image = tf.numpy_function(
        func=_py_load_image,
        inp=[image_path],
        Tout=tf.float32,
    )
    # Set static shape so Keras knows the spatial dimensions and channels.
    image.set_shape((img_height, img_width, 3))
    return image


def make_dataset_from_df(
    df: pd.DataFrame,
    img_height: int,
    img_width: int,
    batch_size: int,
    shuffle: bool = False,
) -> tf.data.Dataset:
    """Create a ``tf.data.Dataset`` from a labels DataFrame.

    Args:
        df: DataFrame containing ``image_path`` and target column
            ``price_usd``.
        img_height: Target image height for resizing.
        img_width: Target image width for resizing.
        batch_size: Number of samples per batch.
        shuffle: Whether to shuffle the dataset at each epoch.

    Returns:
        A ``tf.data.Dataset`` yielding batches of ``(image, target)``.
    """
    required_cols = {"image_path", "price_usd"}
    if not required_cols.issubset(df.columns):
        raise KeyError(
            "DataFrame must contain 'image_path' and 'price_usd' columns."
        )

    image_paths = df["image_path"].astype(str).to_numpy()
    targets = df["price_usd"].to_numpy(dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, targets))

    def _map_fn(
        path: tf.Tensor,
        target: tf.Tensor,
    ) -> Tuple[tf.Tensor, tf.Tensor]:
        image = load_and_preprocess_image(
            image_path=path,
            img_height=img_height,
            img_width=img_width,
        )
        return image, target

    autotune = tf.data.AUTOTUNE
    ds = ds.map(_map_fn, num_parallel_calls=autotune)

    if shuffle:
        buffer_size = max(len(df), batch_size * 4)
        ds = ds.shuffle(
            buffer_size=buffer_size,
            reshuffle_each_iteration=True,
        )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(autotune)

    return ds


# ---------------------------------------------------------------------------
# Build datasets from the split DataFrames
# ---------------------------------------------------------------------------
train_ds = make_dataset_from_df(
    df=df_train,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_ds = make_dataset_from_df(
    df=df_val,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = make_dataset_from_df(
    df=df_test,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# ---------------------------------------------------------------------------
# Quick sanity check
# ---------------------------------------------------------------------------
sample_images, sample_targets = next(iter(train_ds))
print("[INFO] Sample batch shapes:")
print("  images:", sample_images.shape)
print("  targets:", sample_targets.shape)

[WARN] Failed to load image '/content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed/train/images/80098038_052b4d06590a0b5660f416a08cf5bb13-p_f.jpg': image file is truncated (2 bytes not processed)
[INFO] Sample batch shapes:
  images: (32, 224, 224, 3)
  targets: (32,)


- `train_ds`, `val_ds`, and `test_ds` are now:
  - Batched
  - Prefetched
  - Ready for GPU-accelerated training
- Image tensors have shape `(batch_size, 224, 224, 3)`.
- Targets are continuous regression values (e.g., normalised prices).

Next, we define a **baseline CNN model** and train it to establish a
performance reference.

---
# 4. Baseline CNN Model Definition

Before hyperparameter tuning, we establish a **baseline CNN** to:

- Validate that the input pipeline and targets behave as expected.
- Provide a reference performance (MAE / RMSE / R²).

In this section, we:

- Define a custom RMSE metric.
- Implement `build_baseline_cnn()` which returns a compiled Keras model
  suitable for regression on house prices.

In [ ]:
"""Baseline CNN model and custom metrics for CV house price regression."""


def root_mean_squared_error(
    y_true: tf.Tensor,
    y_pred: tf.Tensor,
) -> tf.Tensor:
    """Compute the Root Mean Squared Error (RMSE) metric.

    Args:
        y_true: Ground-truth target tensor.
        y_pred: Predicted target tensor.

    Returns:
        A scalar tensor representing the RMSE between ``y_true`` and
        ``y_pred``.
    """
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true)))


def build_baseline_cnn(
    img_height: int,
    img_width: int,
) -> tf.keras.Model:
    """Build and compile a baseline CNN regression model.

    The architecture is intentionally simple and lightweight, serving as a
    baseline for comparison with more advanced CNN or hybrid models.

    Architecture:
        • Conv2D(32) + MaxPooling2D
        • Conv2D(64) + MaxPooling2D
        • Conv2D(128) + MaxPooling2D
        • Flatten
        • Dense(256, relu) + Dropout(0.3)
        • Dense(1, linear)

    Args:
        img_height: Input image height in pixels.
        img_width: Input image width in pixels.

    Returns:
        A compiled Keras Model configured for regression with:
            • MSE loss
            • MAE metric
            • RMSE metric (custom)
    """
    inputs = tf.keras.Input(
        shape=(img_height, img_width, 3),
        name="image_input",
    )

    x = tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation="relu",
        padding="same",
        name="conv_32",
    )(inputs)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2),
        name="pool_32",
    )(x)

    x = tf.keras.layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        activation="relu",
        padding="same",
        name="conv_64",
    )(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2),
        name="pool_64",
    )(x)

    x = tf.keras.layers.Conv2D(
        filters=128,
        kernel_size=(3, 3),
        activation="relu",
        padding="same",
        name="conv_128",
    )(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2),
        name="pool_128",
    )(x)

    x = tf.keras.layers.Flatten(name="flatten")(x)
    x = tf.keras.layers.Dense(
        256,
        activation="relu",
        name="dense_256",
    )(x)
    x = tf.keras.layers.Dropout(
        rate=0.3,
        name="dropout_0_3",
    )(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="linear",
        name="price_output",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="baseline_cnn",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            root_mean_squared_error,
        ],
    )

    model.summary(print_fn=lambda line: print("[BASELINE] " + line))

    return model


# ---------------------------------------------------------------------------
# Instantiate baseline model
# ---------------------------------------------------------------------------
baseline_model = build_baseline_cnn(
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
)

[BASELINE] Model: "baseline_cnn"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_32 (Conv2D)                │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_32 (MaxPooling2D)          │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_64 (Conv2D)                │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_64 (MaxPooling2D)          │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────

- A simple but functional CNN has been defined:
  - 3 convolutional blocks with max-pooling.
  - Dense layer with dropout for regularisation.
  - Linear output neuron for regression.
- The model is compiled with:
  - Loss: MSE
  - Metrics: MAE and RMSE

We now train this baseline model and record its learning dynamics.

---
# 5. Baseline CNN Training

We now train the baseline CNN using:

- Early stopping on validation loss.
- Learning rate reduction on plateau.
- Model checkpointing to save the best baseline model.

This helps us avoid overfitting and provides a stable reference model.

In [ ]:
"""Training configuration and callbacks for the baseline CNN model.

This section:
    • Configures output directory for baseline artefacts.
    • Defines EarlyStopping, ReduceLROnPlateau, and ModelCheckpoint callbacks.
    • Trains the baseline CNN on the image dataset.
"""

from __future__ import annotations

import os

import tensorflow as tf


# ---------------------------------------------------------------------------
# Baseline artefact directory
# ---------------------------------------------------------------------------
BASELINE_DIR: str = os.path.join(MODEL_ROOT_DIR, "baseline_cnn")
os.makedirs(BASELINE_DIR, exist_ok=True)

BASELINE_CHECKPOINT_PATH: str = os.path.join(
    BASELINE_DIR,
    "baseline_cnn_best.keras",
)

# ---------------------------------------------------------------------------
# Callback definitions
# ---------------------------------------------------------------------------
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
)

reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_delta=0.0,
    min_lr=1e-6,
    verbose=1,
)

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=BASELINE_CHECKPOINT_PATH,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
)


def train_baseline_cnn(
    model: tf.keras.Model,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    epochs: int = 30,
) -> tf.keras.callbacks.History:
    """Train the baseline CNN with standard callbacks.

    Args:
        model: Compiled Keras CNN regression model.
        train_ds: Training dataset of (image, target) batches.
        val_ds: Validation dataset of (image, target) batches.
        epochs: Maximum number of training epochs.

    Returns:
        A Keras History object containing the training and validation metrics
        for each epoch.
    """
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[early_stop_cb, reduce_lr_cb, checkpoint_cb],
    )
    return history


# ---------------------------------------------------------------------------
# Execute training
# ---------------------------------------------------------------------------
EPOCHS_BASELINE: int = 30

baseline_history = train_baseline_cnn(
    model=baseline_model,
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=EPOCHS_BASELINE,
)

Epoch 1/30
[WARN] Failed to load image '/content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed/train/images/80098038_052b4d06590a0b5660f416a08cf5bb13-p_f.jpg': image file is truncated (2 bytes not processed)
332/332 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 245184610304.0000 - mae: 300096.8750 - root_mean_squared_error: 445294.4062
Epoch 1: val_loss improved from inf to 182662512640.00000, saving model to /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/baseline_cnn/baseline_cnn_best.keras
332/332 ━━━━━━━━━━━━━━━━━━━━ 165s 388ms/step - loss: 245040693248.0000 - mae: 299966.7188 - root_mean_squared_error: 445151.9375 - val_loss: 182662512640.0000 - val_mae: 229415.5938 - val_root_mean_squared_error: 362259.4375 - learning_rate: 0.0010
Epoch 2/30
[WARN] Failed to load image '/content/drive/MyDrive/Colab Notebooks/DOAA/cv_data_processed/train/images/80098038_052b4d06590a0b5660f416a08cf5bb13-p_f.jpg': image file is truncated (2 bytes not processed)
329/332 ━━━━━━━━━━━━━━━━━━━━ 

- The baseline model has been trained with:
  - Early stopping and learning rate scheduling.
  - Best weights saved to: `baseline_cnn_best.keras`.
- The training history (`baseline_history`) will be used to:
  - Plot learning curves.
  - Compare against tuned models later.

Next, we will save the baseline training history and plots to avoid
retraining for future analysis.

---
# 6. Persist Baseline Training History and Curves

To ensure we do not need to rerun baseline training:

- We save the `history.history` dictionary as a JSON file.
- We generate and save loss/metric plots (PNG) for:
  - Training vs validation MSE loss
  - Training vs validation MAE
  - Training vs validation RMSE

All artefacts are stored in the `baseline_cnn` directory in Google Drive.

In [ ]:
"""Utilities for saving training history and plotting learning curves."""


def save_history_and_plots(
    history: tf.keras.callbacks.History,
    out_dir: str,
    prefix: str,
) -> None:
    """Save training history to JSON and generate learning curve plots.

    This function:
        • Serialises the history.history dictionary to a JSON file.
        • Generates and saves line plots for key metrics over epochs.

    Args:
        history: Keras History object returned by ``model.fit()``.
        out_dir: Directory where artefacts (JSON + PNG plots) will be stored.
        prefix: Prefix for output filenames (for example, ``"baseline"`` or
            ``"tuned"``).

    Returns:
        None. Files are written directly to ``out_dir``.
    """
    os.makedirs(out_dir, exist_ok=True)

    history_dict: Dict[str, List[float]] = history.history

    # -----------------------------------------------------------------------
    # Save history as JSON
    # -----------------------------------------------------------------------
    history_path = os.path.join(out_dir, f"{prefix}_history.json")
    with open(history_path, "w", encoding="utf-8") as fp:
        json.dump(history_dict, fp, indent=2)
    print(f"[INFO] Saved history JSON → {history_path}")

    # -----------------------------------------------------------------------
    # Helper to plot a single metric
    # -----------------------------------------------------------------------
    def _plot_metric(metric_name: str, ylabel: str) -> None:
        """Plot a single metric and its validation counterpart.

        Args:
            metric_name: Name of the metric in history.history (for example,
                ``"loss"``, ``"mae"``, ``"root_mean_squared_error"``).
            ylabel: Label to display on the y-axis of the plot.

        Returns:
            None. Saves the plot as a PNG file in ``out_dir``.
        """
        train_values = history_dict.get(metric_name)
        val_values = history_dict.get(f"val_{metric_name}")

        if train_values is None or val_values is None:
            print(
                f"[WARN] Metric '{metric_name}' or 'val_{metric_name}' "
                "missing from history; skipping plot."
            )
            return

        epochs = range(1, len(train_values) + 1)
        plt.figure()
        plt.plot(epochs, train_values, label=f"train_{metric_name}")
        plt.plot(epochs, val_values, label=f"val_{metric_name}")
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(f"{prefix}: {metric_name} over epochs")
        plt.legend()
        plot_path = os.path.join(out_dir, f"{prefix}_{metric_name}.png")
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        print(f"[INFO] Saved plot → {plot_path}")

    # -----------------------------------------------------------------------
    # Generate plots for key metrics
    # -----------------------------------------------------------------------
    _plot_metric("loss", "MSE Loss")
    _plot_metric("mae", "Mean Absolute Error")
    _plot_metric("root_mean_squared_error", "Root Mean Squared Error")


# ---------------------------------------------------------------------------
# Save baseline history and plots
# ---------------------------------------------------------------------------
save_history_and_plots(
    history=baseline_history,
    out_dir=BASELINE_DIR,
    prefix="baseline",
)

[INFO] Saved history JSON → /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/baseline_cnn/baseline_history.json
[INFO] Saved plot → /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/baseline_cnn/baseline_loss.png
[INFO] Saved plot → /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/baseline_cnn/baseline_mae.png
[INFO] Saved plot → /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/baseline_cnn/baseline_root_mean_squared_error.png


- Baseline training history has been exported as JSON.
- Learning curves for loss, MAE, and RMSE have been saved as PNG plots.
- All artefacts are stored under:
  - `cv_models/baseline_cnn/`

We now proceed to **hyperparameter tuning** using Keras Tuner to search
for a stronger CNN architecture.

---
# 7. Hyperparameter Tuning with Keras Tuner (Bayesian Optimisation)

To improve on the baseline CNN, we use **Keras Tuner** with a
Bayesian optimisation strategy.

In this section, we:

- Define a `build_cnn_hypermodel()` function that:
  - Varies convolutional filter counts, dense units, dropout rates,
    and learning rate.
- Instantiate a `BayesianOptimization` tuner with:
  - Objective: minimise validation RMSE.
  - Search space defined by the hypermodel.
  - Logs stored under `cv_models/tuner_cv_cnn/` in Google Drive.

In [ ]:
def build_cnn_hypermodel(hp: kt.HyperParameters) -> tf.keras.Model:
    """Build a powerful ResNet-style CNN regression model for tuning.

    The search space includes:
        - Number of residual stages.
        - Number of residual blocks per stage.
        - Filters per stage.
        - L2 weight regularisation strength.
        - Dense head width and optional second dense layer.
        - Dropout rate.
        - Adam learning rate.

    The architecture is inspired by modern ResNet / VGG-style backbones:
    stacked convolutional stages with residual connections, followed by
    global average pooling and a dense regression head.

    Args:
        hp: Keras Tuner HyperParameters object.

    Returns:
        Compiled Keras Model instance for image-based regression.
    """
    # ------------------------------------------------------------------
    # Regularisation
    # ------------------------------------------------------------------
    l2_lambda = hp.Float(
        "l2_lambda",
        min_value=1e-5,
        max_value=1e-3,
        sampling="log",
    )
    kernel_regularizer = tf.keras.regularizers.l2(l2_lambda)

    inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))

    # Optionally add a small input Conv stem (like ResNet/VGG).
    stem_filters = hp.Int(
        "stem_filters",
        min_value=32,
        max_value=96,
        step=32,
    )
    x = tf.keras.layers.Conv2D(
        filters=stem_filters,
        kernel_size=(7, 7),
        strides=(2, 2),
        padding="same",
        use_bias=False,
        kernel_regularizer=kernel_regularizer,
        kernel_initializer="he_normal",
        name="stem_conv",
    )(inputs)
    x = tf.keras.layers.BatchNormalization(name="stem_bn")(x)
    x = tf.keras.layers.Activation("relu", name="stem_relu")(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(3, 3),
        strides=(2, 2),
        padding="same",
        name="stem_pool",
    )(x)

    # ------------------------------------------------------------------
    # Residual block helper (ResNet-style)
    # ------------------------------------------------------------------
    def residual_block(
        block_input: tf.Tensor,
        filters: int,
        stride: int,
        block_name: str,
    ) -> tf.Tensor:
        """Build a single residual block with optional downsampling."""
        shortcut = block_input

        # First conv
        x_rb = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=(3, 3),
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_regularizer=kernel_regularizer,
            kernel_initializer="he_normal",
            name=f"{block_name}_conv1",
        )(block_input)
        x_rb = tf.keras.layers.BatchNormalization(
            name=f"{block_name}_bn1",
        )(x_rb)
        x_rb = tf.keras.layers.Activation(
            "relu",
            name=f"{block_name}_relu1",
        )(x_rb)

        # Second conv
        x_rb = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=(3, 3),
            strides=1,
            padding="same",
            use_bias=False,
            kernel_regularizer=kernel_regularizer,
            kernel_initializer="he_normal",
            name=f"{block_name}_conv2",
        )(x_rb)
        x_rb = tf.keras.layers.BatchNormalization(
            name=f"{block_name}_bn2",
        )(x_rb)

        # Project shortcut if spatial or channel dims do not match.
        if (stride != 1) or (shortcut.shape[-1] != filters):
            shortcut = tf.keras.layers.Conv2D(
                filters=filters,
                kernel_size=(1, 1),
                strides=stride,
                padding="same",
                use_bias=False,
                kernel_regularizer=kernel_regularizer,
                kernel_initializer="he_normal",
                name=f"{block_name}_proj",
            )(shortcut)
            shortcut = tf.keras.layers.BatchNormalization(
                name=f"{block_name}_proj_bn",
            )(shortcut)

        # Add & ReLU
        x_rb = tf.keras.layers.Add(name=f"{block_name}_add")(
            [x_rb, shortcut]
        )
        x_rb = tf.keras.layers.Activation(
            "relu",
            name=f"{block_name}_out",
        )(x_rb)

        return x_rb

    # ------------------------------------------------------------------
    # Residual stages (super powerful body)
    # ------------------------------------------------------------------
    num_stages = hp.Int("num_stages", min_value=3, max_value=5, step=1)
    x_body = x

    for stage_idx in range(num_stages):
        # Filters grow with depth (like ResNet).
        base_filters = hp.Int(
            f"filters_stage_{stage_idx}",
            min_value=64 * (stage_idx + 1),
            max_value=128 * (stage_idx + 1),
            step=32,
        )

        num_blocks = hp.Int(
            f"blocks_stage_{stage_idx}",
            min_value=1,
            max_value=3,
            step=1,
        )

        for block_idx in range(num_blocks):
            stride = 2 if (block_idx == 0 and stage_idx > 0) else 1
            block_name = f"stage{stage_idx}_block{block_idx}"
            x_body = residual_block(
                block_input=x_body,
                filters=base_filters,
                stride=stride,
                block_name=block_name,
            )

    # Global average pooling (ResNet / modern CNN style).
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x_body)

    # ------------------------------------------------------------------
    # Dense head
    # ------------------------------------------------------------------
    dense_units_1 = hp.Int(
        "dense_units_1",
        min_value=128,
        max_value=512,
        step=64,
    )
    x = tf.keras.layers.Dense(
        dense_units_1,
        activation="relu",
        kernel_regularizer=kernel_regularizer,
        name="dense_1",
    )(x)

    use_second_dense = hp.Boolean("use_second_dense")
    if use_second_dense:
        dense_units_2 = hp.Int(
            "dense_units_2",
            min_value=64,
            max_value=dense_units_1,
            step=64,
        )
        x = tf.keras.layers.Dense(
            dense_units_2,
            activation="relu",
            kernel_regularizer=kernel_regularizer,
            name="dense_2",
        )(x)

    dropout_rate = hp.Float(
        "dropout_rate",
        min_value=0.2,
        max_value=0.6,
        step=0.1,
    )
    x = tf.keras.layers.Dropout(dropout_rate, name="dropout")(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="linear",
        name="price_output",
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="resnet_style_cnn",
    )

    # Tunable learning rate.
    learning_rate = hp.Float(
        "learning_rate",
        min_value=1e-4,
        max_value=5e-3,
        sampling="log",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae", root_mean_squared_error],
    )

    return model

TUNER_DIR = os.path.join(MODEL_ROOT_DIR, "tuner_cv_cnn")
os.makedirs(TUNER_DIR, exist_ok=True)

tuner = kt.BayesianOptimization(
    hypermodel=build_cnn_hypermodel,
    objective=kt.Objective(
        "val_root_mean_squared_error",
        direction="min",
    ),
    max_trials=200,
    directory=TUNER_DIR,
    project_name="cv_house_price_tuning",
    overwrite=False,  # Reuse previous results if rerun
)

print("[INFO] Tuner search space summary:")
tuner.search_space_summary()


def get_and_save_best_hyperparameters(
    tuner_obj: kt.BayesianOptimization,
    out_dir: str,
    num_trials: int = 1,
    filename: str = "best_hyperparameters.json",
) -> kt.HyperParameters:
    """Retrieve, print, and save the best hyperparameters from a tuner.

    This function should be called after ``tuner.search`` has completed.

    Args:
        tuner_obj: A configured and run Keras Tuner instance.
        out_dir: Directory where the best hyperparameters JSON will be saved.
        num_trials: Number of top trials to consider; the first is returned.
        filename: Name of the JSON file to write.

    Returns:
        The best HyperParameters object from the tuner.
    """
    os.makedirs(out_dir, exist_ok=True)

    best_hps = tuner_obj.get_best_hyperparameters(num_trials=num_trials)
    best_hp = best_hps[0]

    print("[INFO] Best hyperparameters found:")
    for name, value in best_hp.values.items():
        print(f"  {name}: {value}")

    best_hp_path = os.path.join(out_dir, filename)
    with open(best_hp_path, "w", encoding="utf-8") as fp:
        json.dump(best_hp.values, fp, indent=2)

    print(f"[INFO] Best hyperparameters saved to: {best_hp_path}")

    return best_hp

Reloading Tuner from /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/tuner_cv_cnn/cv_house_price_tuning/tuner0.json
[INFO] Tuner search space summary:
Search space summary
Default search space size: 18
l2_lambda (Float)
{'default': 1e-05, 'conditions': [], 'min_value': 1e-05, 'max_value': 0.001, 'step': None, 'sampling': 'log'}
stem_filters (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 96, 'step': 32, 'sampling': 'linear'}
num_stages (Int)
{'default': None, 'conditions': [], 'min_value': 3, 'max_value': 5, 'step': 1, 'sampling': 'linear'}
filters_stage_0 (Int)
{'default': None, 'conditions': [], 'min_value': 64, 'max_value': 128, 'step': 32, 'sampling': 'linear'}
blocks_stage_0 (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 3, 'step': 1, 'sampling': 'linear'}
filters_stage_1 (Int)
{'default': None, 'conditions': [], 'min_value': 128, 'max_value': 256, 'step': 32, 'sampling': 'linear'}
blocks_stage_1 (Int)
{'default': None, 'condit

- The hypermodel defines a search over:
  - 2–4 convolutional blocks
  - 32–128 filters per block
  - 128–512 dense units
  - 0.2–0.5 dropout
  - 1e-4 to 1e-3 learning rate
- Keras Tuner logs are stored at:
  - `cv_models/tuner_cv_cnn/`

Next, we will run the hyperparameter search on the training data and
validate on the validation split.

---
# 8. Running Keras Tuner Search

We now launch the hyperparameter search using the tuner:

- Uses the `train_ds` for training.
- Uses `val_ds` for validation.
- Applies early stopping to avoid overfitting and reduce wasted trials.
- All trial results are saved automatically to Google Drive via the
  tuner directory.


In [ ]:
tuner_early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=False,
)

EPOCHS_TUNING: int = 50

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TUNING,
    callbacks=[tuner_early_stop_cb],
)

print("[INFO] Tuning completed. Best models summary:")
tuner.results_summary()

# Retrieve and persist best hyperparameters
best_hp = get_and_save_best_hyperparameters(
    tuner_obj=tuner,
    out_dir=TUNER_DIR,
    num_trials=1,
)

[INFO] Tuning completed. Best models summary:
Results summary
Results in /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/tuner_cv_cnn/cv_house_price_tuning
Showing 10 best trials
Objective(name="val_root_mean_squared_error", direction="min")

Trial 048 summary
Hyperparameters:
l2_lambda: 2.086632535304697e-05
stem_filters: 64
num_stages: 3
filters_stage_0: 64
blocks_stage_0: 2
filters_stage_1: 192
blocks_stage_1: 1
filters_stage_2: 352
blocks_stage_2: 3
dense_units_1: 320
use_second_dense: False
dropout_rate: 0.30000000000000004
learning_rate: 0.0021469283898863604
filters_stage_3: 384
blocks_stage_3: 2
dense_units_2: 64
filters_stage_4: 480
blocks_stage_4: 3
Score: 359647.125

Trial 196 summary
Hyperparameters:
l2_lambda: 4.13295792876421e-05
stem_filters: 64
num_stages: 4
filters_stage_0: 64
blocks_stage_0: 2
filters_stage_1: 256
blocks_stage_1: 3
filters_stage_2: 256
blocks_stage_2: 3
dense_units_1: 128
use_second_dense: True
dropout_rate: 0.5
learning_rate: 0.005
filters_stag

- Keras Tuner has evaluated multiple CNN configurations.
- The best trials and their metrics are saved under:
  - `cv_models/tuner_cv_cnn/`
- A summary of the top-performing configurations has been printed.

Next, we will retrieve the **best hyperparameters**, rebuild the best
model, train it more thoroughly, and persist all artefacts.

---
# 9. Training the Best Hyperparameter CNN

In this section, we:

- Retrieve the best hyperparameters from the tuner.
- Rebuild the best CNN model using `build_cnn_hypermodel()`.
- Train this best model with:
  - Early stopping
  - Learning rate reduction
  - Checkpointing
- Save:
  - The best tuned model
  - Training history (JSON)
  - Learning curves (PNG)

In [ ]:
# Directory for tuned model artefacts
TUNED_DIR = os.path.join(MODEL_ROOT_DIR, "tuned_cnn")
os.makedirs(TUNED_DIR, exist_ok=True)

tuned_checkpoint_path = os.path.join(TUNED_DIR, "tuned_cnn_best.keras")

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("[INFO] Best hyperparameters found by tuner:")
for name, value in best_hp.values.items():
    print(f"  {name}: {value}")

tuned_model = build_cnn_hypermodel(best_hp)
tuned_model.summary(print_fn=lambda line: print("[TUNED] " + line))

tuned_early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)

tuned_reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

tuned_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=tuned_checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
)

EPOCHS_TUNED_FINAL: int = 100

tuned_history = tuned_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TUNED_FINAL,
    callbacks=[tuned_early_stop_cb, tuned_reduce_lr_cb, tuned_checkpoint_cb],
)

save_history_and_plots(
    history=tuned_history,
    out_dir=TUNED_DIR,
    prefix="tuned",
)

[INFO] Best hyperparameters found by tuner:
  l2_lambda: 2.086632535304697e-05
  stem_filters: 64
  num_stages: 3
  filters_stage_0: 64
  blocks_stage_0: 2
  filters_stage_1: 192
  blocks_stage_1: 1
  filters_stage_2: 352
  blocks_stage_2: 3
  dense_units_1: 320
  use_second_dense: False
  dropout_rate: 0.30000000000000004
  learning_rate: 0.0021469283898863604
  filters_stage_3: 384
  blocks_stage_3: 2
  dense_units_2: 64
  filters_stage_4: 480
  blocks_stage_4: 3


[TUNED] Model: "resnet_style_cnn"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │      9,408 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu           │ (None, 112, 112,  

- The best-performing CNN configuration from Keras Tuner has been:
  - Rebuilt
  - Trained with stronger stopping criteria
  - Saved to: `tuned_cnn_best.keras`
- Training history and plots (loss, MAE, RMSE) have been exported to:
  - `cv_models/tuned_cnn/`

We now evaluate the tuned model on the **held-out test set** and save
the final performance metrics.

---
# 10. Final Evaluation on Test Set and Metrics Export

The final step is to:

- Evaluate the tuned CNN on the held-out `test_ds`.
- Compute regression metrics (loss, MAE, RMSE).
- Save these metrics into a JSON file for reporting and future reference.

In [ ]:
def evaluate_and_save_metrics(
    model: tf.keras.Model,
    test_dataset: tf.data.Dataset,
    out_dir: str,
    filename: str = "test_metrics.json",
) -> Dict[str, float]:
    """Evaluate a model on the test dataset and save metrics to JSON.

    Args:
        model: Trained Keras model to be evaluated.
        test_dataset: tf.data.Dataset for testing.
        out_dir: Directory to store the metrics file.
        filename: Name of the JSON file to write.

    Returns:
        Dictionary of metric names to their scalar values.
    """
    os.makedirs(out_dir, exist_ok=True)

    results = model.evaluate(test_dataset, verbose=1)
    metric_names = model.metrics_names

    metrics_dict: Dict[str, float] = {
        name: float(value) for name, value in zip(metric_names, results)
    }

    metrics_path = os.path.join(out_dir, filename)
    with open(metrics_path, "w", encoding="utf-8") as fp:
        json.dump(metrics_dict, fp, indent=2)

    print(f"[INFO] Saved test metrics → {metrics_path}")
    print("[INFO] Test metrics:")
    for name, value in metrics_dict.items():
        print(f"  {name}: {value:.4f}")

    return metrics_dict


# Evaluate tuned model on held-out test set
test_metrics = evaluate_and_save_metrics(
    model=tuned_model,
    test_dataset=test_ds,
    out_dir=TUNED_DIR,
)

72/72 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - loss: 179387088896.0000 - mae: 220353.9844 - root_mean_squared_error: 431199.1562
[INFO] Saved test metrics → /content/drive/MyDrive/Colab Notebooks/DOAA/cv_models/tuned_cnn/test_metrics.json
[INFO] Test metrics:
  loss: 175466201088.0000
  compile_metrics: 217769.5000


- The tuned CNN has been evaluated on the test set.
- Key metrics (loss, MAE, RMSE) have been saved in:
  - `cv_models/tuned_cnn/test_metrics.json`
- All major artefacts are now safely persisted in Google Drive:
  - Baseline and tuned models
  - Histories (JSON)
  - Learning curve plots (PNG)
  - Hyperparameter tuning logs

This notebook can now be referenced in your DOAA report as the
**Computer Vision Modelling Pipeline**, and future runs can reuse
the saved artefacts without retraining from scratch.